# Long-Only ML Strategy Notebook

This notebook is organized for a repeatable workflow:
1. Setup
2. Build data/features
3. Train models (pre-2021 cutoff)
4. Build 5-year scoring panel (2021-2025)
5. Define strategy/helpers
6. Run 5-year frozen OOS RL policy training
7. First backtest: Frozen 5-year OOS (no retraining)
8. Run anchored vs frozen 5-year OOS comparison


## A) Setup


In [1]:
import os
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modules.data.context import DataContext
from modules.data.universe_fmp import build_large_liquid_universe_single_call
from modules.api import (
    build_technical_dataframe,
    build_fundamental_dataframe,
    build_macro_dataframe,
    build_label_dataframe,
)
from modules.data.preparation import MLDatasetConfig, prepare_ml_dataset
from modules.workflows import (
    train_rf_models,
    train_ae,
    StrategyCase,
    ProbabilityColumnConfig,
    enrich_scored_panel,
    make_backtest_panel,
    make_exec_cfg,
    run_case as run_eval_case,
    strategy_diagnostics,
    build_anchored_fold,
    RLConfig,
    run_a2c_workflow,
)
from modules.engine.latest import run_panel_prediction_custom, make_autoencoder_familiarity_predictor
from modules.engine.backtest import backtest_panel
from modules.strategies.benchmark import BuyAndHoldEqualWeightStrategy

pd.set_option("display.max_columns", 200)

# ------------------------------
# Global Config (single source of truth)
# ------------------------------
APP_CFG = {
    "dates": {
        "train_cutoff": "2020-12-31",
        "bt_start": "2021-01-01",
        "bt_end": "2025-12-31",
        "data_start": "1980-01-01",
    },
    "universe": {
        "size": None,  # None => full screener universe
    },
    "runtime": {
        "cache_enabled": True,
        "fast_rl_metrics": True,
    },
    "costs": {
        "fee_bps": 5.0,
        "slippage_bps": 5.0,
    },
    "strategy": {
        "top_k": 20,
        "rebalance_freq": "W",   # D, W, M
        "gate_q": 0.50,
        "gross": 0.8,
    },
    "labels": {
        "k_params": {"YE": [1, 2, 4, 8]},
        "use_sample_weight": True,
        "alpha": 4.0,
        "r_clip": 0.10,
        "horizon_balance": True,
    },
    "probability_columns": {
        "buy_col": "clf__prob_1",
        "short_col": None,
        "infer_short_from_buy": True,
    },
    "rl": {
        "lookback_window": 10,
        "initial_balance": 100000.0,
        "ppo_seed": 42,
        "ppo_episodes": 30,
        "drawdown_penalty_lambda": 0.10,
        "force_buy_sell_everything": False,
    },
}



## B) Build Features


In [2]:
# 1) Context + universe + feature panel
load_dotenv()
FMP_API_KEY = os.getenv("FMP_API_KEY")
if not FMP_API_KEY:
    raise RuntimeError("Missing FMP_API_KEY in environment/.env")

ctx = DataContext.from_data_dir(
    api_key=FMP_API_KEY,
    data_dir="./data",
    db_name="quant.db",
    sleep_s=0.0,
    history_years=100,
)

universe = tuple(build_large_liquid_universe_single_call(
    api_key=FMP_API_KEY,
    marketCapMoreThan=10_000_000_000,
    priceMoreThan=5,
    country="US",
    exchange="NASDAQ,NYSE,AMEX",
    isEtf=False,
    isFund=False,
    includeAllShareClasses=False,
    limit=10000,
))

if APP_CFG["universe"]["size"] is not None:
    universe = tuple(universe[: int(APP_CFG["universe"]["size"])])

universe = universe
if len(universe) == 0:
    raise RuntimeError("Screener returned 0 symbols.")

START_DATE = APP_CFG["dates"]["data_start"]
END_DATE = pd.Timestamp(APP_CFG["dates"]["bt_end"]).strftime("%Y-%m-%d")

technical_df, technical_cols = build_technical_dataframe(
    ctx=ctx,
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
    verbose_data=False,
)
fund_df, fund_cols = build_fundamental_dataframe(
    ctx=ctx,
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
    target_index=technical_df.index,
    daily_prices=technical_df,
    verbose=False,
)
macro_df, macro_cols = build_macro_dataframe(
    ctx=ctx,
    start_date=START_DATE,
    end_date=END_DATE,
    target_index=technical_df.index,
    verbose=False,
)

final_df = pd.concat([technical_df, fund_df, macro_df], axis=1).sort_index()

print("Universe size:", len(universe))
print("Feature panel shape:", final_df.shape)
print("Feature date range:", final_df.index.get_level_values("date").min().date(), "->", final_df.index.get_level_values("date").max().date())


[WARNING] Skipped 9 symbols during build.


/Users/johnnylee/PycharmProjects/optimal_trader/modules/features/fundamentals_fmp.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fund_df = pd.concat(dfs_per_symbol, ignore_index=True)


Universe size: 789
Feature panel shape: (4031649, 202)
Feature date range: 1997-05-12 -> 2025-12-31


## C) Train Base Models (Pre-2021 Cutoff)


In [3]:
# 2) Train models using data <= train_cutoff (pre-bt)
TRAIN_CUTOFF_TS = pd.Timestamp(APP_CFG["dates"]["train_cutoff"])
train_mask = final_df.index.get_level_values("date") <= TRAIN_CUTOFF_TS
features_train = final_df.loc[train_mask].copy()

symbols_in_train = set(technical_df.loc[technical_df.index.get_level_values("date") <= TRAIN_CUTOFF_TS].index.get_level_values("symbol"))
daily_map_train = {
    s: technical_df.xs(s, level="symbol").loc[:TRAIN_CUTOFF_TS].copy()
    for s in universe
    if s in symbols_in_train
}

K_PARAMS = dict(APP_CFG["labels"]["k_params"])
EXECUTION_PARAMS = {
    "price_col": "close",
    "fee_bps": float(APP_CFG["costs"]["fee_bps"]),
    "slippage_bps": float(APP_CFG["costs"]["slippage_bps"]),
}
WEIGHTING_PARAMS = {
    "use_sample_weight": bool(APP_CFG["labels"]["use_sample_weight"]),
    "alpha": float(APP_CFG["labels"]["alpha"]),
    "r_clip": float(APP_CFG["labels"]["r_clip"]),
    "horizon_balance": bool(APP_CFG["labels"]["horizon_balance"]),
}

label_df_train = build_label_dataframe(
    daily_by_symbol=daily_map_train,
    k_params=K_PARAMS,
    execution_params=EXECUTION_PARAMS,
    weighting=WEIGHTING_PARAMS,
    add_rank_labels=True,
    verbose=False,
)

train_df, raw_feature_list, _ = prepare_ml_dataset(
    features_df=features_train,
    labels_df=label_df_train,
    target_cols=["target", "trade_return", "trade_duration_days"],
    weight_col="sample_weight",
    config=MLDatasetConfig(drop_nan_features=False),
    verbose=True,
)

rf_bundle = train_rf_models(
    train_df,
    raw_feature_list,
    split_ratio=1.0,
    classifier_target_col="target",
    ranking_target_col="rank_y",
    classifier_market_position_col=None,
    train_trade_return_model=True,
    trade_return_target_col="trade_return",
    train_duration_model=False,
)

clf_raw = rf_bundle.clf
reg_raw = rf_bundle.trade_return_reg if rf_bundle.trade_return_reg is not None else rf_bundle.ranking_reg

ae_raw, ae_numeric_cols = train_ae(train_df, raw_feature_list)

print("Train rows:", len(train_df))
print("Train date max:", pd.to_datetime(train_df.index.get_level_values('date')).max().date())


--- Preparing ML Training Dataset ---
  - Final Training Rows: 398,968
  - Active Features:     202
  - Targets:             ['target', 'trade_return', 'trade_duration_days']
  - Sample Weight Col:   sample_weight

  DIAGNOSTIC: SKLEARN RANDOM FOREST CLASSIFIER (TEST RESULTS)
DATASET & SPLIT:
  - Total Observations: 398,968
  - Split Mode:         In-sample eval (no internal holdout split)
  - Features:           197 numeric (filtered 5 strings)

CLASS DISTRIBUTION (Mapping: {0: '0', 1: '1'}):
               Train Set              Test Set
  -        0: 199,498 (50.0%)   199,498 (50.0%)
  -        1: 199,470 (50.0%)   199,470 (50.0%)

TEST PERFORMANCE:
  - Accuracy:           99.08%
  - ROC AUC:            0.9995

CONFUSION MATRIX (Test Set):
            Pred       0 Pred       1 
True       0:       197969         1529 
True       1:         2158       197312 

TOP 10 FEATURES:
  - ZClose10: 0.1095
  - BBPos10: 0.1071
  - DistSMA10: 0.0991
  - PosInChannel10: 0.0617
  - DistEMA12: 0.0

## D) Build Scoring Panels (History + 2021-2025 OOS)



In [ ]:
# 3) Score panel for model backtest + RL (with pre-cutoff history)
BT_START_TS = pd.Timestamp(APP_CFG["dates"]["bt_start"])
BT_END_TS = pd.Timestamp(APP_CFG["dates"]["bt_end"])
DATA_START_TS = pd.Timestamp(APP_CFG["dates"]["data_start"])
prob_cfg = ProbabilityColumnConfig(**APP_CFG["probability_columns"])

panel_for_scoring = final_df.loc[
    (final_df.index.get_level_values("date") >= DATA_START_TS)
    & (final_df.index.get_level_values("date") <= BT_END_TS)
].copy()

if panel_for_scoring.empty:
    raise RuntimeError("No feature rows in scoring panel.")

ae_predict = make_autoencoder_familiarity_predictor(ae_numeric_cols)

scored_panel_all = run_panel_prediction_custom(
    train_data=panel_for_scoring,
    model_specs=[
        {"model": clf_raw, "pred_col": "clf", "include_class_probs": True},
        {"model": reg_raw, "pred_col": "ranking"},
        {"model": ae_raw, "pred_col": "ae_familiarity", "predict_fn": lambda df, m: ae_predict(df, m)},
    ],
    market_position_value=None,
    combine_scores_fn=lambda df: pd.to_numeric(df[prob_cfg.buy_col], errors="coerce").fillna(0.0)
    * pd.to_numeric(df["ranking"], errors="coerce").fillna(0.0)
    * pd.to_numeric(df["ae_familiarity"], errors="coerce").fillna(1.0),
    row_filter_fn=None,
    round_decimals=None,
)

scored_panel_all = enrich_scored_panel(scored_panel_all, prob_config=prob_cfg)
bt_panel_all = make_backtest_panel(
    scored_panel=scored_panel_all,
    technical_df=technical_df,
    start=DATA_START_TS,
    end=BT_END_TS,
)

bt_panel_5y = bt_panel_all.loc[
    (bt_panel_all.index.get_level_values("date") >= BT_START_TS)
    & (bt_panel_all.index.get_level_values("date") <= BT_END_TS)
].copy()

print("Backtest panel shape (all years for RL train/eval):", bt_panel_all.shape)
print("Backtest panel shape (5-year OOS display):", bt_panel_5y.shape)
print(
    "Backtest date range:",
    bt_panel_all.index.get_level_values("date").min().date(),
    "->",
    bt_panel_all.index.get_level_values("date").max().date(),
)


## E) Strategy And Backtest Config (Shared)



In [ ]:
# 4) Strategy/backtest config (module-native)

cfg = make_exec_cfg(
    fee_bps=float(APP_CFG["costs"]["fee_bps"]),
    slippage_bps=float(APP_CFG["costs"]["slippage_bps"]),
    execution_mode="rl_env",
)

print("Execution config:")
display(cfg)


## F) Diagnostics (Optional)



In [ ]:
# 5) Diagnostics note

print(
    "Diagnostics are available via modules.workflows.strategy_diagnostics(strategy, panel). "
    "Run it on any strategy object if you need exposure/turnover diagnostics."
)


## G) RL Policy Training (Train <= Cutoff, Eval > Cutoff)



In [ ]:
# 6) RL policy training via workflow module

RL_CFG = {
    **dict(APP_CFG["rl"]),
    "years": list(range(pd.Timestamp(APP_CFG["dates"]["bt_start"]).year, pd.Timestamp(APP_CFG["dates"]["bt_end"]).year + 1)),
    "train_split_date": APP_CFG["dates"]["train_cutoff"],
    "rebalance_freq": APP_CFG["strategy"]["rebalance_freq"],
    "max_stocks_per_day": int(APP_CFG["strategy"]["top_k"]),
    "eligibility_quantile": float(APP_CFG["strategy"]["gate_q"]),
    "fee_bps": float(APP_CFG["costs"]["fee_bps"]),
    "slippage_bps": float(APP_CFG["costs"]["slippage_bps"]),
}

FROZEN_BT_YEARS = list(RL_CFG["years"])

rl_config = RLConfig(
    lookback_window=int(RL_CFG["lookback_window"]),
    eligibility_quantile=float(RL_CFG["eligibility_quantile"]),
    rebalance_freq=RL_CFG["rebalance_freq"],
    max_stocks_per_day=RL_CFG["max_stocks_per_day"],
    initial_balance=float(RL_CFG["initial_balance"]),
    fee_bps=float(RL_CFG["fee_bps"]),
    slippage_bps=float(RL_CFG["slippage_bps"]),
    ppo_episodes=int(RL_CFG["ppo_episodes"]),
    drawdown_penalty_lambda=float(RL_CFG["drawdown_penalty_lambda"]),
    seed=int(RL_CFG["ppo_seed"]),
    force_buy_sell_everything=bool(RL_CFG.get("force_buy_sell_everything", False)),
)

rl_results = run_a2c_workflow(
    bt_panel=bt_panel_all.copy(),
    cfg=rl_config,
    train_split_date=pd.Timestamp(RL_CFG["train_split_date"]),
    years=FROZEN_BT_YEARS,
)

print(
    f"A2C training done: steps/episode={rl_results['train_steps_per_episode']} | "
    f"episodes={rl_config.ppo_episodes} | total_timesteps={rl_results['total_timesteps']}"
)
print("Executed trade counts (filled orders only):")
display(rl_results["executed_action_counts"].rename("count"))
print("Policy action counts (all dates x symbols):")
display(rl_results["action_counts"].rename("count"))
print("Latest day actions:")
display(rl_results["last_actions"].head(30))


## H) RL OOS Backtest Results (Post-Cutoff)



In [ ]:
# 7) RL backtest results (from workflow module)

symbols = rl_results["symbols"]
policy_action_type = rl_results["policy_action_type"]
E_seq = rl_results["eligible_by_day"]
B_seq = rl_results["buy_score_by_day"]
D_seq = rl_results["dates"]
REBALANCE_MASK = rl_results["rebalance_mask"]
close_bt = rl_results["close_bt"]

rl_combined_eq = rl_results["rl_equity"]
rl_combined_ret = rl_results["rl_returns"]
rl_cash = rl_results["rl_cash"]
rl_yearly_df = rl_results["rl_yearly_df"]
first_backtest_summary = rl_results["rl_summary_df"]

print("RL yearly OOS results (per-stock framework backtest):")
display(rl_yearly_df)
print("RL combined OOS summary:")
display(first_backtest_summary)

fig, ax = plt.subplots(figsize=(12, 4))
rl_combined_eq.plot(ax=ax, lw=2, color="darkgreen")
ax.set_title(f"RL Agent 5-Year OOS Equity ({FROZEN_BT_YEARS[0]}-{FROZEN_BT_YEARS[-1]}) | Discrete Entry/Exit + Dynamic 1/n Sizing")
ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.grid(True, alpha=0.3)
plt.show()

display(pd.DataFrame({
    "equity": rl_combined_eq,
    "returns": rl_combined_ret,
    "cash": rl_cash,
}).tail())

print(f"Executed trade log rows: {len(rl_results['trade_log'])}")
display(rl_results["trade_log"])

# Keep compatibility placeholders for downstream references.
strategy = None
res = None
best_row = {"case": "rl_agent_policy"}
bt_panel = bt_panel_5y


In [ ]:
# 8) Buy-and-hold benchmark (strategy + unified backtest engine)
bh_strategy = BuyAndHoldEqualWeightStrategy(
    price_col="close",
    gross_exposure=1.0,
    top_k=None,
    liquidate_on_last_day=True,
)

bh_res = backtest_panel(panel=bt_panel_5y, strategy=bh_strategy, cfg=cfg)
bh_eq = bh_res.equity_curve.rename("equity")
bh_ret = bh_res.returns.rename("returns")

# Align benchmark to RL evaluation dates for apples-to-apples comparison tables.
bh_eq = bh_eq.reindex(rl_combined_eq.index).ffill()
bh_ret = bh_ret.reindex(rl_combined_ret.index).fillna(0.0)

bh_total_return_pct = float((bh_eq.iloc[-1] / bh_eq.iloc[0] - 1.0) * 100.0) if len(bh_eq) else np.nan
bh_sharpe = float((bh_ret.mean() / bh_ret.std(ddof=0)) * np.sqrt(252.0)) if bh_ret.std(ddof=0) > 1e-12 else np.nan
bh_mdd = float((((bh_eq / bh_eq.cummax()) - 1.0).min()) * 100.0) if len(bh_eq) else np.nan

bh_yearly_rows = []
for yr in FROZEN_BT_YEARS:
    yret = bh_ret.loc[(bh_ret.index >= pd.Timestamp(f"{yr}-01-01")) & (bh_ret.index <= pd.Timestamp(f"{yr}-12-31"))]
    yeq = (1.0 + yret).cumprod()
    y_total = float((yeq.iloc[-1] - 1.0) * 100.0) if len(yeq) else np.nan
    y_sharpe = float((yret.mean() / yret.std(ddof=0)) * np.sqrt(252.0)) if len(yret) and yret.std(ddof=0) > 1e-12 else np.nan
    y_mdd = float((((yeq / yeq.cummax()) - 1.0).min()) * 100.0) if len(yeq) else np.nan
    bh_yearly_rows.append({
        "mode": "buy_and_hold_equal_weight_strategy",
        "test_year": int(yr),
        "total_return_pct": y_total,
        "sharpe": y_sharpe,
        "max_drawdown_pct": y_mdd,
    })

bh_yearly_df = pd.DataFrame(bh_yearly_rows)
print("Buy-and-hold yearly OOS results (strategy):")
display(bh_yearly_df)

comparison_df = pd.DataFrame([
    {
        "mode": "rl_agent_a2c",
        "total_return_pct": float(first_backtest_summary.iloc[0]["combined_total_return_pct"]),
        "sharpe": float(first_backtest_summary.iloc[0]["combined_sharpe"]),
        "max_drawdown_pct": float(first_backtest_summary.iloc[0]["combined_max_drawdown_pct"]),
    },
    {
        "mode": "buy_and_hold_equal_weight_strategy",
        "total_return_pct": bh_total_return_pct,
        "sharpe": bh_sharpe,
        "max_drawdown_pct": bh_mdd,
    },
])
print("RL vs Buy-and-Hold comparison (combined 5-year):")
display(comparison_df)

# Plot normalized equity so both curves are comparable on one axis.
rl_norm = rl_combined_eq / max(float(rl_combined_eq.iloc[0]), 1e-12)
bh_norm = bh_eq / max(float(bh_eq.iloc[0]), 1e-12)

fig, ax = plt.subplots(figsize=(12, 4))
rl_norm.plot(ax=ax, lw=2, label="RL (normalized)", color="darkgreen")
bh_norm.plot(ax=ax, lw=2, label="Buy & Hold (normalized)", color="steelblue")
ax.set_title(f"RL vs Buy-and-Hold Normalized Equity ({FROZEN_BT_YEARS[0]}-{FROZEN_BT_YEARS[-1]})")
ax.set_xlabel("Date")
ax.set_ylabel("Growth (start=1.0)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

display(pd.DataFrame({
    "rl_equity": rl_combined_eq,
    "bh_equity": bh_eq,
    "rl_returns": rl_combined_ret,
    "bh_returns": bh_ret,
}).tail())
